# Replay a TerraPulse measurement
Kathmandu saved 199-pixel footprint. This offline notebook contains three derived HH means on common-valid native support, with actual UTC and source granule IDs. No raw imagery or credentials are included.
Source data: NASA–ISRO NISAR GCOV, ASF DAAC, DOI [10.5067/NIL2GCOV-P1](https://doi.org/10.5067/NIL2GCOV-P1). This replay uses derived TerraPulse expanded-v1 footprint means. The [NASA catalog entry](https://data.nasa.gov/dataset/nisar-provisional-geocoded-polarimetric-covariance-product-version-1-d9e04) identifies the dataset and its data-use terms.


In [ ]:
from math import isclose
sample = [{'utc': '2026-08-19T23:39:18Z', 'source': 'NISAR_L2_PR_GCOV_028_098_A_016_4005_DHDH_A_20260819T233918_20260819T233953_P05023_N_F_J_001', 'hh_mean_db': -4.770070468438274}, {'utc': '2026-08-31T23:39:18Z', 'source': 'NISAR_L2_PR_GCOV_029_098_A_016_4005_DHDH_A_20260831T233918_20260831T233952_P05023_N_F_J_001', 'hh_mean_db': -12.296379446384297}, {'utc': '2026-09-12T23:39:18Z', 'source': 'NISAR_L2_PR_GCOV_030_098_A_016_4005_DHDH_A_20260912T233918_20260912T233952_P05023_N_F_J_001', 'hh_mean_db': -11.84668738877953}]
reference, previous, selected = sample
recent = selected['hh_mean_db'] - previous['hh_mean_db']
persistent = selected['hh_mean_db'] - reference['hh_mean_db']
print(f"Selected UTC: {selected['utc']}")
print(f"Recent: {recent:+.2f} dB; reference: {persistent:+.2f} dB")
assert isclose(recent, 0.44969205760476605, abs_tol=1e-10)
assert isclose(persistent, -7.076616920341256, abs_tol=1e-10)


## Check your exported dossier

Place a TerraPulse investigation JSON beside this notebook and set `DOSSIER` to its filename. In Colab, upload the JSON to the notebook session first. The cell checks the signed arithmetic and the date bindings in the exported trace. It uses the saved measurements and requires only the Python standard library.

The sample above uses precomputed means. Re-measuring native pixels requires the matching TerraPulse local runtime and raster masks.


In [ ]:
from pathlib import Path
import json

DOSSIER = ''  # Set to the filename of an exported investigation.

def check_trace(path):
    packet = json.loads(Path(path).read_text())
    assert packet['schema'] == 'terrapulse.atlas-investigation.v2'
    selected = packet['contract']['currentId']
    reference = packet['contract']['referenceId']
    history = packet['history']
    by_id = {row['observation_id']: row for row in history}
    assert len(by_id) == len(history)
    assert by_id[selected]['timestamp'] == packet['selected_utc']
    assert by_id[reference]['timestamp'] == packet['reference_utc']
    checked = 0
    for row in history:
        previous_id = row['comparison_endpoints']['recent']
        if previous_id is None:
            continue
        assert row['comparison_endpoints']['reference'] == reference
        previous = by_id[previous_id]
        assert previous['timestamp'] < row['timestamp']
        for channel in ('hh', 'hv'):
            a, b = (v[channel + '_reference'] for v in (row, previous))
            expected = row[channel + '_successive']
            if a is None or b is None or expected is None:
                continue
            assert isclose(a - b, expected, rel_tol=0, abs_tol=1e-6), (row['timestamp'], channel)
            checked += 1
    return packet['study'], packet['footprint'], packet['selected_utc'], checked

if DOSSIER:
    print(check_trace(DOSSIER))
